# Aula 20 — Agentic Systems Foundations

**Quando a próxima ação deixa de ser fixa?**

Na Aula 18, aprendemos workflows determinísticos.  
Na Aula 19, aprendemos a expor e descobrir capabilities via MCP.

Agora entra uma nova variável:

> **autonomia na escolha da próxima ação.**

Regra da aula:

```text
não chamar tudo de agente
→ tornar a decisão observável
→ comparar com alternativa mais simples
→ medir benefício, custo e risco
```

## 1. Objetivos

Ao final, você deverá conseguir:

- distinguir workflow determinístico de agentic loop;
- localizar onde a autonomia aparece;
- separar decisão, autorização e execução;
- explicar o papel do agent harness como camada de runtime e controle;
- implementar estado, action space, step budget e termination conditions;
- observar human escalation;
- comparar workflow e agentic loop sob os mesmos cenários;
- justificar quando autonomia adicional não compensa.

## 📘 Glossário da aula

Conceitos centrais:

- **[Loop agente](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#loop-agente)**
- **[Autonomia](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#autonomia)**
- **[Orçamento de passos](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#orçamento-de-passos)**
- **[Condição de término](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#condição-de-término)**
- **[Escalonamento humano](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#escalonamento-humano)**
- **[Agent Harness](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#agent-harness)**

Use o glossário durante o laboratório, não apenas como referência final.

## 2. Tool, workflow, MCP e agente não são sinônimos

```text
Tool
→ executa uma capacidade

Workflow
→ coordena capacidades sob transições explícitas

MCP
→ padroniza exposição e discovery

Agentic loop
→ observa estado
→ escolhe próxima ação
→ executa sob controles
→ observa resultado
→ decide novamente
```

Um workflow pode ter `if`, branches, retries e checkpoints e ainda continuar sendo determinístico.

A pergunta decisiva é:

> **Quem ou o que escolhe a próxima ação a partir do estado observado?**

### Comparando os quatro conceitos

| Conceito | Executa capacidade | Coordena passos | Escolhe próxima ação em runtime | Padroniza integração |
|---|---:|---:|---:|---:|
| **Tool** | sim | não | não | não |
| **Workflow** | sim | sim | normalmente não | não |
| **MCP** | indiretamente | não | não | sim |
| **Agentic loop** | sim | sim | sim | pode usar MCP |

A tabela ajuda a evitar uma confusão comum:

```text
ter tools
≠
ter workflow
≠
usar MCP
≠
ser agentic
```

O ponto distintivo da Aula 20 é a coluna:

> **Escolhe próxima ação em runtime?**

Mesmo assim, essa escolha continua limitada por action space, governance e termination policies.

### Mini exercício — Onde existe comportamento agentic?

Classifique mentalmente cada caso antes de revelar a solução.

**Caso A**

```python
if score < 0.70:
    human_review()
else:
    approve()
```

**Caso B**

```text
state
→ decision_provider(state)
→ proposed_action
→ governance_gate
→ execute
```

**Caso C**

Um sistema consulta um servidor MCP para descobrir tools disponíveis e, depois, executa sempre a mesma sequência fixa.

Perguntas:

1. Qual caso possui branching sem necessariamente possuir comportamento agentic?
2. Qual caso apresenta escolha explícita da próxima ação em runtime?
3. Usar MCP transforma automaticamente um sistema em agente?

In [ ]:
concept_solution = """Caso A: branching determinístico; não é automaticamente agentic.
Caso B: apresenta comportamento agentic porque a próxima ação é proposta a partir do estado observado.
Caso C: MCP não cria agência; o sistema pode continuar sendo um workflow determinístico."""

print("Depois de responder, execute a próxima linha manualmente se quiser revelar a solução:")
print("print(concept_solution)")

## 3. Arquitetura de referência

```text
Goal
 ↓
Observed State
 ↓
Decision Provider
 ↓
Proposed Action
 ↓
Governance Gate
 ↓
Execution
 ↓
Observation
 ↓
State Update
 ↺
```

Importante:

```text
decisão ≠ autorização
decisão ≠ execução
agentic loop ≠ loop ilimitado
```

Nesta aula, o decision provider será determinístico e local. Isso nos permite estudar **agência sem confundir agência com LLM**.

### 3.1 Agent Harness — a camada de runtime ao redor da decisão

A arquitetura acima pode ser vista como uma implementação mínima de um **agent harness**.

Nesta aula, usamos o termo de forma operacional:

> **Agent Harness é a camada executável que envolve o decision provider e controla como estado, capabilities, políticas, observações, limites e terminação participam da execução.**

Uma forma simples de visualizar:

```text
                 Decision Provider
                        ↓
┌─────────────────────────────────────────┐
│              AGENT HARNESS              │
│                                         │
│  state                                  │
│  action space                           │
│  governance gate                        │
│  capability / tool execution            │
│  observation                            │
│  state update                           │
│  step budget                            │
│  termination                            │
│  human escalation                       │
│  observability                          │
└─────────────────────────────────────────┘
                        ↓
                   Environment
```

Isso ajuda a separar três ideias:

```text
Model / Decision Provider
→ produz proposta ou decisão

Harness
→ controla como a decisão entra no runtime

Agentic behavior
→ emerge da combinação entre decisão, estado, ação, observação e iteração
```

Portanto:

```text
LLM ≠ agent
tool ≠ agent
MCP ≠ agent
harness ≠ model
```

O harness também não é sinônimo de framework. Um framework pode fornecer componentes para construir um harness; o harness é a camada efetiva de execução e controle do sistema.

### Harness, MCP, workflow e orchestrator

Esses termos podem coexistir:

| Conceito | Papel principal |
|---|---|
| **MCP** | padronizar exposição e descoberta de capacidades |
| **Workflow** | coordenar uma sequência ou política de execução |
| **Agent Harness** | controlar o runtime do comportamento agente |
| **Orchestrator** | coordenar múltiplos componentes, fluxos ou agentes |

Um agent harness pode **usar MCP** para acessar capabilities, pode executar **workflows** internos e pode futuramente participar de uma arquitetura com **orchestrators**.

A pergunta pedagógica continua a mesma:

> **Qual camada decide, qual camada governa e qual camada executa?**

In [ ]:
from dataclasses import dataclass, field, asdict
from copy import deepcopy
import pandas as pd

ACTIONS = {
    "inspect_request",
    "get_lesson_status",
    "build_release_note",
    "request_approval",
    "publish_simulated",
    "ask_human",
    "stop",
}

SCENARIOS = {
    "ready_approved": {
        "goal": "publicar release da aula 20",
        "request_valid": True,
        "lesson_status": "ready",
        "approval_available": True,
    },
    "not_ready": {
        "goal": "publicar release da aula 20",
        "request_valid": True,
        "lesson_status": "review-candidate",
        "approval_available": True,
    },
    "ambiguous": {
        "goal": "faça o que for melhor com a aula",
        "request_valid": None,
        "lesson_status": "ready",
        "approval_available": True,
    },
    "approval_missing": {
        "goal": "publicar release da aula 20",
        "request_valid": True,
        "lesson_status": "ready",
        "approval_available": False,
    },
}

print("Cenários:", list(SCENARIOS))
print("Action space:", sorted(ACTIONS))

### O que observar nesta saída

A primeira célula executável torna dois contratos visíveis:

```text
SCENARIOS
→ quais situações o laboratório vai testar

ACTIONS
→ quais ações o sistema pode propor e executar
```

O `action space` é **fechado**. Isso significa que o agente não ganha novas capacidades por “criatividade”: ele só pode escolher entre ações previamente expostas e governadas.

Essa distinção é importante:

```text
autonomia de decisão
≠
autonomia ilimitada de capacidade
```

O sistema pode decidir **qual ação permitida usar**, mas não pode inventar uma tool nova nem executar algo fora da allow-list.

## 4. Estado explícito

O agente não deve depender de “memória invisível”.

O estado carregará apenas informação operacional necessária para a decisão.

In [ ]:
@dataclass
class AgentState:
    goal: str
    request_valid: object = None
    lesson_status: object = None
    approval_available: bool = False
    inspected: bool = False
    release_note: object = None
    approval: object = None
    published: bool = False
    needs_human: bool = False
    done: bool = False
    step_count: int = 0
    history: list = field(default_factory=list)
    termination_reason: object = None

def initial_state(scenario):
    # Fair comparison: lesson_status is NOT preloaded.
    # Both architectures must obtain it through the same capability.
    return AgentState(
        goal=scenario["goal"],
        request_valid=scenario["request_valid"],
        lesson_status=None,
        approval_available=scenario["approval_available"],
    )

### O que o `state` representa?

O `AgentState` funciona como a **fonte operacional de verdade** do loop.

Ele não é “memória secreta” do modelo e não é chain-of-thought. É um objeto explícito que registra fatos relevantes para decidir a próxima ação.

Exemplos:

- `lesson_status=None` → ainda não observamos o status;
- `approval=None` → ainda não sabemos se há aprovação;
- `published=True` → o objetivo operacional foi atingido;
- `done=True` → o loop deve terminar.

A consequência é importante:

> **Se uma decisão depende de um fato, esse fato deve estar representado ou ser obtido de forma observável.**

Isso melhora auditabilidade, teste e reprodução do comportamento.

## 5. Capabilities locais

As capabilities são deliberadamente simples. Nenhuma produz side effect externo real.

In [ ]:
def execute_action(action, state, scenario):
    if action == "inspect_request":
        state.inspected = True
        return {"request_valid": state.request_valid}

    if action == "get_lesson_status":
        state.lesson_status = scenario["lesson_status"]
        return {"lesson_status": state.lesson_status}

    if action == "build_release_note":
        note = f"Release simulada — status={state.lesson_status}"
        state.release_note = note
        return {"release_note": note}

    if action == "request_approval":
        state.approval = bool(state.approval_available)
        return {"approval": state.approval}

    if action == "publish_simulated":
        state.published = True
        return {"published": True}

    if action == "ask_human":
        state.needs_human = True
        state.done = True
        state.termination_reason = "human_escalation"
        return {"needs_human": True}

    if action == "stop":
        state.done = True
        state.termination_reason = state.termination_reason or "safe_stop"
        return {"done": True}

    raise ValueError(f"Ação desconhecida: {action}")

### Capability ≠ Decision

Uma capability responde à pergunta:

> **“Eu sei fazer isso?”**

Ela não responde:

> **“Isso é o que devo fazer agora?”**

Exemplo:

```text
get_lesson_status
→ sabe obter o status

build_release_note
→ sabe construir a nota

publish_simulated
→ sabe simular a publicação
```

Nenhuma dessas capabilities decide quando deve ser chamada.

A separação arquitetural é:

```text
capability
→ sabe fazer

decision provider
→ escolhe o que fazer

governance
→ verifica se pode fazer

executor
→ realiza a ação
```

Essa distinção evita colocar decisão, autorização e execução dentro da mesma função.

## 6. Baseline — workflow determinístico

Primeiro, resolvemos o problema com uma sequência conhecida.

Esse baseline é importante: **não podemos justificar autonomia sem comparar com algo mais simples**.

In [ ]:
def run_workflow(scenario):
    state = initial_state(scenario)
    trace = []
    sequence = [
        "inspect_request",
        "get_lesson_status",
        "build_release_note",
        "request_approval",
        "publish_simulated",
    ]

    for step, action in enumerate(sequence, start=1):
        if action == "inspect_request" and state.request_valid is not True:
            state.needs_human = True
            state.done = True
            state.termination_reason = "human_escalation"
            break

        if action == "build_release_note" and state.lesson_status != "ready":
            state.done = True
            state.termination_reason = "lesson_not_ready"
            break

        if action == "publish_simulated" and state.approval is not True:
            state.needs_human = True
            state.done = True
            state.termination_reason = "human_escalation"
            break

        result = execute_action(action, state, scenario)
        trace.append({"step": step, "action": action, "result": result})

    if state.published:
        state.done = True
        state.termination_reason = "goal_completed"

    return state, trace

wf_state, wf_trace = run_workflow(SCENARIOS["ready_approved"])
display(pd.DataFrame(wf_trace))
print("Termination:", wf_state.termination_reason)

### O que observar no baseline

No cenário `ready_approved`, o workflow executa:

```text
inspect_request
→ get_lesson_status
→ build_release_note
→ request_approval
→ publish_simulated
```

e termina com:

```text
Termination: goal_completed
```

O ponto pedagógico é que a **ordem das ações já foi definida pelo programador**.

Mesmo existindo condições e interrupções, a política de execução continua explícita no código.

Pergunta para o aluno:

> Onde está a decisão sobre a próxima ação?

No workflow, ela está majoritariamente codificada na própria sequência e nas regras condicionais. O sistema não possui um componente separado cuja responsabilidade seja escolher dinamicamente a próxima ação a partir do estado.

## 7. Decision Provider — onde a agência entra

Agora a próxima ação não vem de uma lista fixa.

O provider recebe o estado atual e propõe **uma ação dentre as permitidas**.

Ele é determinístico nesta aula para tornar a lógica auditável.

In [ ]:
def choose_next_action(state):
    if state.done:
        return "stop", "estado já terminal"

    if state.request_valid is None:
        return "ask_human", "solicitação ambígua"

    if state.request_valid is False:
        return "stop", "solicitação inválida"

    if not state.inspected:
        return "inspect_request", "pedido ainda não foi inspecionado"

    if state.lesson_status is None:
        return "get_lesson_status", "status da aula ainda não foi observado"

    if state.lesson_status != "ready":
        return "stop", "aula ainda não está pronta"

    if state.release_note is None:
        return "build_release_note", "release note ainda não existe"

    if state.approval is None:
        return "request_approval", "publicação exige aprovação"

    if state.approval is False:
        return "ask_human", "aprovação não disponível"

    if not state.published:
        return "publish_simulated", "pré-condições atendidas"

    return "stop", "objetivo concluído"

### Onde a autonomia aparece

Aqui surge a principal diferença arquitetural da aula.

`choose_next_action(state)` recebe o estado observado e devolve:

```text
ação proposta
+
razão operacional resumida
```

Exemplo conceitual:

```text
lesson_status is None
→ propose get_lesson_status
→ "status da aula ainda não foi observado"
```

A função **não executa** a ação. Ela apenas propõe.

Isso preserva três responsabilidades separadas:

```text
Decision Provider
→ o que proponho fazer?

Governance Gate
→ isso pode ser feito?

Executor
→ execute a ação permitida
```

A coluna `decision_reason` usada depois no trace é uma justificativa operacional curta e auditável. Ela **não representa raciocínio privado do modelo**.

## 8. Governance Gate

A proposta do decision provider não executa automaticamente.

Antes de executar, verificamos:

- action allow-list;
- step budget;
- política para publicação;
- estado incompatível.

Isso preserva:

> **decisão do agente não é autorização.**

In [ ]:
def governance_gate(action, state, max_steps):
    if action not in ACTIONS:
        return False, "action_not_allowed"

    if state.step_count >= max_steps:
        return False, "step_budget_exceeded"

    if action == "publish_simulated" and state.approval is not True:
        return False, "publish_requires_approval"

    return True, "allowed"

### Por que existe um Governance Gate?

O decision provider pode propor uma ação, mas a proposta não se autoriza sozinha.

O gate verifica, entre outros pontos:

```text
ação pertence à allow-list?
step budget foi excedido?
publicação possui aprovação?
```

Portanto:

> **decisão ≠ autorização ≠ execução**

Essa separação permite que a autonomia seja limitada por política externa ao decision provider.

Mesmo que futuramente o provider seja um LLM, a regra continua a mesma: o modelo pode propor; a política decide se a ação é admissível.

## 9. Agentic loop mínimo

Observe o ciclo:

```text
estado
→ decisão
→ gate
→ execução
→ observação
→ novo estado
```

O histórico registra **ações e razões resumidas**, não raciocínio privado do modelo.

In [ ]:
def run_agentic(scenario, max_steps=7):
    state = initial_state(scenario)
    trace = []

    while not state.done:
        action, reason = choose_next_action(state)
        allowed, gate_reason = governance_gate(action, state, max_steps)

        before = {
            "step_count": state.step_count,
            "lesson_status": state.lesson_status,
            "published": state.published,
            "needs_human": state.needs_human,
            "approval": state.approval,
        }

        if not allowed:
            state.done = True
            state.termination_reason = gate_reason
            trace.append({
                "step": state.step_count + 1,
                "action": action,
                "decision_reason": reason,
                "governance": gate_reason,
                "result": None,
                "state_before": before,
            })
            break

        state.step_count += 1
        result = execute_action(action, state, scenario)

        if action == "stop" and state.termination_reason == "safe_stop":
            if state.lesson_status != "ready":
                state.termination_reason = "lesson_not_ready"
            elif state.published:
                state.termination_reason = "goal_completed"

        if state.published:
            state.done = True
            state.termination_reason = "goal_completed"

        trace.append({
            "step": state.step_count,
            "action": action,
            "decision_reason": reason,
            "governance": gate_reason,
            "result": result,
            "state_before": before,
        })

    return state, trace

agent_state, agent_trace = run_agentic(SCENARIOS["ready_approved"])
display(pd.DataFrame(agent_trace))
print("Termination:", agent_state.termination_reason)

### Como ler o trace do agentic loop

A tabela do loop deve ser lida horizontalmente:

```text
state_before
→ decision_reason
→ action
→ governance
→ result
```

No cenário `ready_approved`, vemos uma cadeia semelhante ao workflow, mas agora cada transição passa por uma decisão explícita.

Isso é crucial:

```text
mesma sequência observada
≠
mesma arquitetura
```

O workflow percorre uma política predefinida.

O agentic loop decide cada próximo passo a partir do estado atual, ainda que, neste cenário simples, termine escolhendo a mesma sequência de ações.

### Pergunta guiada — Mesma sequência significa mesma arquitetura?

No cenário `ready_approved`, workflow e agentic loop terminaram executando praticamente a mesma sequência:

```text
inspect_request
→ get_lesson_status
→ build_release_note
→ request_approval
→ publish_simulated
```

Então eles são arquiteturalmente iguais?

**Não.**

No workflow:

```text
a próxima ação
→ vem da estrutura predefinida
```

No agentic loop:

```text
estado observado
→ decision provider
→ próxima ação proposta
```

Portanto:

> **mesma trajetória observada não implica mesmo mecanismo de decisão.**

Esse ponto é fundamental para avaliar sistemas agentes: o comportamento final pode parecer idêntico, mas o locus de decisão, os custos e as superfícies de falha podem ser diferentes.

## 10. Comparative Evidence Lab

Agora comparamos as duas arquiteturas sobre os **mesmos quatro cenários**.

A pergunta não é “qual parece mais inteligente?”.

A pergunta é:

> **Que comportamento adicional a autonomia produziu e qual custo operacional veio junto?**

In [ ]:
def summarize_run(architecture, scenario_name, state, trace):
    actions = [row["action"] for row in trace]
    return {
        "architecture": architecture,
        "scenario": scenario_name,
        "success": bool(state.published),
        "steps": len(trace),
        "decision_count": len(trace) if architecture == "agentic_loop" else 0,
        "tool_calls": sum(a not in {"stop", "ask_human"} for a in actions),
        "human_intervention": bool(state.needs_human),
        "termination_reason": state.termination_reason,
        "cost_proxy": len(trace) + (0.5 * (len(trace) if architecture == "agentic_loop" else 0)),
        "latency_proxy": len(trace),
    }

rows = []

for name, scenario in SCENARIOS.items():
    wf_state, wf_trace = run_workflow(scenario)
    rows.append(summarize_run("workflow", name, wf_state, wf_trace))

    ag_state, ag_trace = run_agentic(scenario)
    rows.append(summarize_run("agentic_loop", name, ag_state, ag_trace))

comparison = pd.DataFrame(rows)
display(comparison)

### Resultado observado

Nos quatro cenários medidos, o agentic loop **não melhorou o desfecho operacional** em relação ao workflow.

Resumo:

| Cenário | Workflow | Agentic loop | Leitura |
|---|---|---|---|
| `ready_approved` | sucesso, 5 passos, custo 5.0 | sucesso, 5 passos, custo 7.5 | mesmo resultado, mais decisões |
| `not_ready` | stop seguro em 2 passos | stop seguro em 3 passos | autonomia adicionou overhead |
| `ambiguous` | human escalation | human escalation | mesmo desfecho, custo maior no loop |
| `approval_missing` | human escalation em 4 passos | human escalation em 5 passos | mesmo desfecho, maior custo |

A evidência observada foi:

```text
mais autonomia
→ mais decisões
→ maior cost_proxy
→ nenhum ganho de task success
```

Isso **não prova que agentes são piores que workflows**.

Prova algo mais específico e mais útil:

> **Para estes cenários, a variabilidade do problema não foi suficiente para justificar a complexidade adicional da autonomia.**

Esse é exatamente o tipo de conclusão que o TIL procura: a arquitetura deve ser escolhida por evidência, não por novidade tecnológica.

### Importante — o que `cost_proxy` significa aqui?

O `cost_proxy` desta aula é uma **métrica didática**, construída apenas para tornar o overhead arquitetural observável.

Ele **não representa** diretamente:

- custo financeiro;
- consumo real de CPU/GPU;
- preço de tokens;
- tempo de rede;
- latência de produção.

Nesta versão:

```text
workflow
→ custo proporcional ao número de passos

agentic loop
→ custo dos passos
+ penalidade didática pelas decisões adicionais
```

Portanto, leia o resultado como:

> **“esta arquitetura adicionou mais trabalho de coordenação?”**

e não como:

> **“esta arquitetura custa exatamente X reais ou X milissegundos.”**

Em sistemas reais, custo e latência devem ser medidos com métricas observadas do ambiente de execução.

### Como interpretar

A comparação é deliberadamente justa:

```text
mesmo objetivo
+ mesmos cenários
+ mesmas capabilities
+ mesma informação inicial
```

Em particular, nenhuma arquitetura recebe o `lesson_status` gratuitamente: ambas precisam obtê-lo por `get_lesson_status`.

Procure três tipos de evidência:

1. **mesmo resultado, mais complexidade** — autonomia não se justificou naquele cenário;
2. **menos trabalho desnecessário ou adaptação ao estado** — a seleção dinâmica de ações trouxe benefício;
3. **novo ponto de falha** — decisão, loop ou policy criaram risco que o workflow não tinha.

Não existe obrigação de o agentic loop vencer.

> **Se o workflow resolver igualmente bem com menor custo e risco, ele é a melhor arquitetura.**

## 11. Failure Lab — step budget

Um agente precisa saber parar.

Vamos criar deliberadamente um provider defeituoso que nunca progride.

In [ ]:
def run_broken_loop(max_steps=3):
    scenario = SCENARIOS["ready_approved"]
    state = initial_state(scenario)
    trace = []

    while not state.done:
        action = "inspect_request"  # provider defeituoso: repete a mesma ação
        allowed, gate_reason = governance_gate(action, state, max_steps)

        if not allowed:
            state.done = True
            state.termination_reason = gate_reason
            break

        state.step_count += 1
        result = execute_action(action, state, scenario)
        trace.append({"step": state.step_count, "action": action, "result": result})

    return state, trace

broken_state, broken_trace = run_broken_loop()
display(pd.DataFrame(broken_trace))
print("Termination:", broken_state.termination_reason)

### Resultado observado do Failure Lab

O provider defeituoso repetiu:

```text
inspect_request
inspect_request
inspect_request
```

e terminou com:

```text
Termination: step_budget_exceeded
```

Observe que:

- a ação era válida;
- a tool funcionava;
- o governance gate permitia a ação;
- nenhum erro de execução ocorreu.

Mesmo assim, o sistema **não progrediu**.

Isso revela uma classe diferente de falha:

```text
ação válida
+ execução válida
+ decisão repetitiva
=
ausência de progresso
```

Por isso, o `step_budget` não é apenas proteção contra um `while` mal escrito.

Ele é um **controle explícito de autonomia e terminação**.

### Progress failure — quando tudo funciona, mas nada avança

O Failure Lab revela uma classe de falha que não é um erro de tool.

Podemos distinguir:

```text
tool failure
→ a ação falhou ao executar

policy failure
→ uma ação indevida foi permitida ou uma ação válida foi bloqueada

decision failure
→ uma ação inadequada foi escolhida

progress failure
→ ações válidas se repetem sem aproximar o sistema do objetivo
```

No exemplo desta aula:

```text
inspect_request
→ inspect_request
→ inspect_request
```

Cada execução é válida isoladamente.

O problema aparece no comportamento do sistema ao longo do tempo.

Isso é um **progress failure**.

### Contenção ≠ recuperação

Quando o `step_budget` interrompe o loop, ele evita execução indefinida.

Mas ele **não resolve** a causa do problema.

```text
step_budget_exceeded
→ contém o dano
→ interrompe a repetição
→ preserva controle

mas não:
→ escolhe nova estratégia
→ corrige o decision provider
→ recupera o objetivo
```

Portanto:

> **detection / containment ≠ recovery**

Essa distinção prepara o caminho para capacidades posteriores como planning, recovery e replanning.

### Interpretação

O `step_budget_exceeded` não é um detalhe de implementação.

Ele é evidência de uma diferença arquitetural:

```text
workflow
→ número de transições conhecido pela estrutura

agentic loop
→ precisa de controles explícitos para evitar não-terminação
```

A autonomia cria capacidade — e também cria uma superfície adicional de governança.

## 12. Architecture Decision Lab

Escolha a arquitetura mínima suficiente:

| Caso | Situação |
|---|---|
| A | sequência estável, baixa variabilidade, alto risco |
| B | poucas branches conhecidas e auditáveis |
| C | objetivo claro, mas ordem das ações depende do estado observado |
| D | tarefa ambígua e irreversível |

Escolha entre:

```text
direct_function
deterministic_workflow
agentic_loop
```

Justifique usando:

- variabilidade;
- previsibilidade;
- reversibilidade;
- custo;
- risco;
- necessidade de supervisão.

> Não escolha `agentic_loop` apenas porque é a opção mais nova.

### Antes de responder

Não escolha a arquitetura mais sofisticada por padrão.

Use esta sequência de perguntas:

```text
1. A ordem das ações é conhecida antecipadamente?
2. Existem poucas branches estáveis e auditáveis?
3. O estado observado pode mudar qual ação deve vir a seguir?
4. O erro é reversível?
5. Quanto custa uma decisão adicional?
6. É necessário human oversight?
```

Uma possível leitura:

- **A** → `deterministic_workflow`;
- **B** → normalmente `deterministic_workflow`;
- **C** → melhor candidato a `agentic_loop`;
- **D** → priorizar supervisão humana; autonomia pode nem ser justificável.

A resposta correta não é “usar agente quando houver complexidade”.

É:

> **usar autonomia quando a decisão em runtime produzir utility suficiente para justificar custo, risco e menor previsibilidade.**

### Matriz de decisão arquitetural

Em vez de escolher por intuição, registre os critérios de cada caso.

Preencha antes de revelar a solução de referência.

In [ ]:
architecture_matrix = pd.DataFrame([
    {
        "case": "A",
        "variability": None,
        "reversibility": None,
        "risk": None,
        "order_known": None,
        "human_oversight": None,
        "choice": None,
    },
    {
        "case": "B",
        "variability": None,
        "reversibility": None,
        "risk": None,
        "order_known": None,
        "human_oversight": None,
        "choice": None,
    },
    {
        "case": "C",
        "variability": None,
        "reversibility": None,
        "risk": None,
        "order_known": None,
        "human_oversight": None,
        "choice": None,
    },
    {
        "case": "D",
        "variability": None,
        "reversibility": None,
        "risk": None,
        "order_known": None,
        "human_oversight": None,
        "choice": None,
    },
])

display(architecture_matrix)

### Solução de referência — interpretação possível

Depois de preencher sua matriz, compare com uma leitura possível:

| Caso | Variabilidade | Reversibilidade | Risco | Ordem conhecida? | Escolha provável |
|---|---|---|---|---|---|
| A | baixa | baixa | alta | sim | deterministic workflow |
| B | baixa/média | alta | média | majoritariamente | deterministic workflow |
| C | alta | média | média | não completamente | agentic loop |
| D | alta | baixa | alta | incerta | human oversight / autonomia restrita |

Não existe obrigação de usar `agentic_loop` em tarefas complexas.

O critério é:

> **a autonomia precisa produzir ganho observável suficiente para justificar custo, risco e menor previsibilidade.**

In [ ]:
from IPython.display import Markdown, display

class TILExercise:
    def __init__(self, hint_text, solution_text):
        self._hint_text = hint_text
        self._solution_text = solution_text

    def hint(self):
        display(Markdown(f"### Dica\n\n{self._hint_text}"))

    def solution(self):
        display(Markdown(f"### Solução de referência\n\n{self._solution_text}"))

q20_arch = TILExercise(
    "Pergunte primeiro se a ordem das ações precisa ser escolhida em runtime. Depois considere risco e reversibilidade.",
    "A tende a deterministic_workflow; B também tende a deterministic_workflow; C é o melhor candidato a agentic_loop; D deve priorizar supervisão humana e pode nem justificar autonomia.",
)

print("Use q20_arch.hint() ou q20_arch.solution() somente quando desejar.")

In [ ]:
# Registre sua decisão antes de revelar a solução.
my_decisions = {
    "A": None,
    "B": None,
    "C": None,
    "D": None,
}

my_decisions

In [ ]:
# Descomente somente depois de tentar.
# q20_arch.solution()

## 13. Checagem de compreensão

Responda sem olhar o código.

### Conceitos

1. Por que um workflow com branches não é automaticamente um agente?
2. Onde a autonomia aparece no laboratório?
3. Por que `choose_next_action()` não deve executar diretamente a tool?
4. O que o governance gate protege?
5. Por que um step budget é parte da arquitetura e não apenas “defensive programming”?
6. Quando `ask_human` é um resultado melhor que continuar executando?

### Evidência observada

7. No cenário `ready_approved`, workflow e agentic loop tiveram o mesmo sucesso. O que, então, diferenciou as arquiteturas?
8. Por que `cost_proxy=7.5` não deve ser interpretado como custo financeiro real?
9. No cenário `not_ready`, o agentic loop precisou de mais passos para chegar ao mesmo desfecho. O que isso revela?
10. No Failure Lab, por que repetir uma ação válida pode ser classificado como `progress failure`?
11. O `step_budget` recuperou o objetivo ou apenas conteve a falha? Explique.

### Decisão arquitetural

12. Em que tipo de problema o workflow determinístico continua sendo preferível?
13. Que evidência você exigiria antes de trocar um workflow por um agentic loop?
14. Como a conclusão desta aula se relaciona com a regra da Aula 13C de que maior capacidade não implica maior utility?

15. O que pertence ao Agent Harness nesta aula e o que permanece responsabilidade do Decision Provider?
16. Por que dizer “Model + Harness” é útil como abstração didática, mas não deve ser tratado como definição universal de agente?

### Respostas de referência — revele somente depois de responder

Use cada bloco como autocorreção. Tente formular sua resposta antes de abrir.

<details>
<summary><strong>1. Por que um workflow com branches não é automaticamente um agente?</strong></summary>

Porque branching apenas expressa caminhos condicionais. Se as transições já foram definidas pela política do workflow, o sistema continua seguindo uma estrutura predefinida. Na Aula 20, o comportamento agentic aparece quando um decision provider escolhe a próxima ação em runtime a partir do estado observado.

</details>

<details>
<summary><strong>2. Onde a autonomia aparece no laboratório?</strong></summary>

Na função `choose_next_action(state)`. Ela recebe o estado atual e propõe a próxima ação entre as ações permitidas. A autonomia é limitada: o provider não inventa capabilities e sua proposta ainda passa pelo governance gate.

</details>

<details>
<summary><strong>3. Por que choose_next_action() não deve executar diretamente a tool?</strong></summary>

Para preservar separação de responsabilidades. O decision provider propõe; o governance gate verifica se a ação é admissível; o executor realiza a ação. Isso evita que quem decide também se autorize e execute sem controle externo.

</details>

<details>
<summary><strong>4. O que o governance gate protege?</strong></summary>

Ele protege a fronteira entre decisão e execução. Nesta aula verifica allow-list, step budget e a política de aprovação antes de publicação. Em sistemas reais, também pode incorporar autorização, schemas, risco, políticas de segurança e outras restrições.

</details>

<details>
<summary><strong>5. Por que um step budget é parte da arquitetura e não apenas defensive programming?</strong></summary>

Porque um sistema com decisão iterativa pode continuar escolhendo ações válidas sem progredir. O step budget define um limite explícito para a autonomia e garante uma condição de término observável.

</details>

<details>
<summary><strong>6. Quando ask_human é um resultado melhor que continuar executando?</strong></summary>

Quando faltam informação, autorização ou condições seguras para agir. Escalar para uma pessoa pode ser o resultado correto quando continuar significaria adivinhar, violar política ou executar uma ação irreversível sob incerteza.

</details>

<details>
<summary><strong>7. No cenário ready_approved, o que diferenciou as arquiteturas se ambas tiveram sucesso?</strong></summary>

O mecanismo de transição. O workflow seguiu uma sequência/política predefinida; o agentic loop escolheu cada próxima ação a partir do estado. O resultado foi igual, mas o agentic loop introduziu decisões adicionais e maior overhead.

</details>

<details>
<summary><strong>8. Por que cost_proxy=7.5 não é custo financeiro real?</strong></summary>

Porque `cost_proxy` foi criado como métrica didática para tornar visível o trabalho adicional de coordenação. Ele não mede reais, tokens, CPU/GPU ou tempo de rede. Custos reais exigem instrumentação do ambiente de execução.

</details>

<details>
<summary><strong>9. O que o cenário not_ready revela?</strong></summary>

Que autonomia pode adicionar overhead mesmo quando o desfecho permanece igual. O agentic loop precisou decidir explicitamente parar, enquanto o workflow chegou ao mesmo estado terminal com menos passos.

</details>

<details>
<summary><strong>10. Por que repetir uma ação válida pode ser progress failure?</strong></summary>

Porque validade local não garante progresso global. `inspect_request` pode executar corretamente a cada chamada, mas repeti-la indefinidamente não aproxima o sistema do objetivo. O problema está na trajetória de decisão, não na tool isolada.

</details>

<details>
<summary><strong>11. O step_budget recuperou o objetivo ou apenas conteve a falha?</strong></summary>

Apenas conteve a falha. Ele interrompeu a repetição e limitou o dano, mas não corrigiu o decision provider nem encontrou uma nova estratégia. Containment não é recovery.

</details>

<details>
<summary><strong>12. Quando o workflow determinístico continua preferível?</strong></summary>

Quando a sequência é conhecida, as branches são estáveis e auditáveis, o risco exige previsibilidade e a autonomia não demonstra ganho suficiente de utility. Nesses casos, a arquitetura mais simples tende a ser mais barata e controlável.

</details>

<details>
<summary><strong>13. Que evidência exigir antes de trocar um workflow por um agentic loop?</strong></summary>

Evidência de que a decisão dinâmica em runtime melhora resultados relevantes — por exemplo, task success, adaptação a variabilidade, redução de ações desnecessárias ou melhor tratamento de exceções — em magnitude suficiente para compensar custo, latência, risco e perda de previsibilidade.

</details>

<details>
<summary><strong>14. Como isso se relaciona com a Aula 13C?</strong></summary>

A Aula 13C ensinou que maior capacidade de modelo não implica maior utility. A Aula 20 estende a mesma regra à arquitetura: maior autonomia também não implica maior utility. Complexidade precisa ser conquistada por evidência.

</details>

<details>
<summary><strong>15. O que pertence ao Agent Harness e o que pertence ao Decision Provider?</strong></summary>

O Decision Provider propõe a próxima ação. O Agent Harness controla o runtime ao redor dessa decisão: estado, action space, governance, execução de capabilities, observação, atualização de estado, step budget, terminação, human escalation e observabilidade.

</details>

<details>
<summary><strong>16. Por que Model + Harness é apenas uma abstração didática?</strong></summary>

Porque sistemas agentes reais podem ter múltiplos modelos, planners, workflows, memória, subagentes, protocolos e componentes externos. A expressão ajuda a separar capacidade decisória de infraestrutura de runtime, mas não pretende definir universalmente o que é um agente.

</details>

## 14. Reprodutibilidade

```text
Internet OFF
GPU OFF
sem API externa
sem side effects reais
decision provider determinístico
action space fechado
step budget explícito
```

O objetivo desta primeira versão é isolar o mecanismo de agência antes de adicionar modelos generativos ou frameworks.

## 15. Síntese

A cadeia da aula foi:

```text
capabilities
→ workflow determinístico
→ estado explícito
→ decision provider
→ governance gate
→ agentic loop
→ observação
→ termination / human escalation
→ comparação de utility
```

### O que a evidência mostrou

Nesta tarefa, o agentic loop funcionou corretamente, mas **não produziu utility adicional suficiente para superar o workflow**.

```text
mais autonomia
→ mais decisões
→ maior overhead
→ mesmo task success
```

Isso reforça uma regra já vista na Aula 13C:

```text
maior capacidade
≠
maior utility
```

Agora aplicada à arquitetura:

```text
maior autonomia
≠
maior utility
```

### Ideia principal

> **Agência não é possuir tools. É introduzir autonomia controlada na escolha da próxima ação a partir do estado observado.**

### Limitação deixada em aberto

O agentic loop desta aula escolhe **uma ação por vez**. Ele não constrói um plano explícito, não representa dependências futuras e não replaneja.

Essa limitação prepara a próxima pergunta curricular:

> **Quando escolher apenas a próxima ação deixa de ser suficiente e passa a valer a pena planejar?**

E a regra do TIL continua valendo:

> **Se um workflow resolve o problema igualmente bem com menor risco e complexidade, prefira o workflow.**

### Harness como síntese arquitetural

A Aula 20 também pode ser resumida assim:

```text
Decision Provider
+
Agent Harness
=
execução agentic controlada
```

Essa expressão é uma abstração didática, não uma fórmula universal.

O valor do harness é tornar explícito que **o comportamento do sistema não depende apenas do modelo**. Estado, políticas, tools, observabilidade, limites e terminação também determinam o resultado.

---

## Continue no TIL

← **[Anterior: Aula 19 — Model Context Protocol (MCP)](https://www.kaggle.com/code/pedrogentil/til-19-model-context-protocol)** &nbsp;&nbsp;|&nbsp;&nbsp; 🏠 **[Apresentação do curso](https://www.kaggle.com/code/pedrogentil/text-intelligence-lab-course)** &nbsp;&nbsp;|&nbsp;&nbsp; **[Próxima: Aula 21 — Planning and Goal Decomposition](https://github.com/pedroregato/text-intelligence-lab/blob/main/course/21-planning-and-goal-decomposition/21-til-planning-and-goal-decomposition.ipynb) — em preparação** →


---

## Continue no TIL

← **[Anterior: Aula 19 — Model Context Protocol (MCP)](https://www.kaggle.com/code/pedrogentil/til-19-model-context-protocol)** &nbsp;&nbsp;|&nbsp;&nbsp; 🏠 **[Apresentação do curso](https://www.kaggle.com/code/pedrogentil/text-intelligence-lab-applied-ai-engineering)** &nbsp;&nbsp;|&nbsp;&nbsp; **[Próxima: Aula 21 — Planning and Goal Decomposition](https://www.kaggle.com/code/pedrogentil/til-21-planning-and-goal-decomposition)** →

TIL · Pedro Gentil Regato de Oliveira Soares · Conteúdo: CC BY 4.0 · Código: MIT
